In [ ]:
# ============================================
# PROPINSIGHT — PROPERTY DOMAIN LABELING PIPELINE (Colab Ready)
# ============================================

# ============== CELL 1: Install Dependencies ==============
!pip install -q openai tiktoken pandas numpy python-dotenv tenacity spacy

# ============== CELL 2: Mount Drive & Paths ==============
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/PropInsight"

RAW_BASE        = f"{BASE}/raw/government_websites"  # raw *.json / *.jsonl
PROCESSED_BASE  = f"{BASE}/processed"                # fallback *.csv
NORMALIZED_DIR  = f"{BASE}/normalized"
RAW_COMBINED    = f"{NORMALIZED_DIR}/government_raw_combined.csv"

OUTPUT_DIR      = f"{BASE}/labeled"                  # output folder
ENTITYRULER     = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
REGEX_JSONL     = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"

# ============== CELL 3: OpenAI API Key ====================
from google.colab import userdata
import os

try:
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key: ')
os.environ['OPENAI_API_KEY'] = api_key

# ============== CELL 4: Config (edit here) ================
MODEL        = "gpt-4o"   # or "gpt-4o" for higher quality
BATCH_SIZE   = 8
MAX_RECORDS  = 0             # ← Start with 100; set to 0 to process ALL rows
AGENCY_ONLY  = ""              # e.g., "HDB", "MND" to filter; empty = all

print("Config:",
      f"\n  MODEL={MODEL}",
      f"\n  MAX_RECORDS={'ALL' if MAX_RECORDS==0 else MAX_RECORDS}",
      f"\n  AGENCY_ONLY={AGENCY_ONLY or 'ALL'}")

# ============== CELL 5: Imports & Shared Helpers ==========
import re, json, time, logging, pandas as pd, numpy as np, hashlib, glob
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime, timezone
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from pathlib import Path
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("propinsight")

def deep_clean_text(text: Any) -> str:
    """Remove HTML, nav, and artifacts."""
    if text is None or (isinstance(text, float) and pd.isna(text)): return ""
    text = str(text)
    # strip HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # common gov site noise
    navigation_patterns = [
        r'Newsroom\s*Press\s*Releases\s*View',
        r'Newsroom\s*Speeches\s*Press\s*Releases\s*Parliament\s*Matters',
        r'About\s*Us\s*Press\s*Releases',
        r'Skip\s*to\s*main\s*content',
        r'Breadcrumb.*?(?=\n)',
        r'Share\s*this\s*page',
        r'Last\s*updated:.*?\d{4}',
        r'©\s*\d{4}.*?rights\s*reserved',
        r'Select\s*Year\s*All',
        r'From\s*To\s*Go',
        r'Read\s*press\s*release',
        r'PrevNext',
    ]
    for pat in navigation_patterns:
        text = re.sub(pat, " ", text, flags=re.I|re.S)
    # URLs/emails
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)
    # collapse punctuation/whitespace
    text = re.sub(r'([.,!?;:]){2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def safe_json_load(s: Any):
    if not isinstance(s, str): return None
    s = s.strip()
    if not s: return None
    try:
        return json.loads(s)
    except Exception:
        try:
            return json.loads(re.sub(r"[‘’“”]", '"', s))
        except Exception:
            return None

def pick_val(d):
    if isinstance(d, dict):
        for k in ["value","sentiment","classification","label","tone"]:
            if k in d and d[k]:
                return str(d[k]).lower()
    if isinstance(d, str): return d.lower()
    return None

def pick_score(d):
    if isinstance(d, dict):
        for k in ["confidence_score","confidence","score","overall_sentiment_confidence","overall_sentiment_confidence_score"]:
            if k in d:
                try: return float(d[k])
                except: pass
    return None

def now_iso():
    return datetime.now(timezone.utc).isoformat()

# ============== CELL 6: Gazetteer & Policy Lexicon =========
# (Property-domain only; Singlish disabled)
GAZETTEER = [
    "Ang Mo Kio","Bishan","Bukit Batok","Bukit Panjang","Bukit Timah",
    "Choa Chu Kang","Clementi","Geylang","Hougang","Jurong East","Jurong West",
    "Kallang","Marine Parade","Pasir Ris","Punggol","Queenstown","Sembawang",
    "Sengkang","Serangoon","Tampines","Toa Payoh","Woodlands","Yishun",
    "Bedok","Boon Lay","Orchard","Marina Bay","Novena","Tanglin","River Valley",
    "Seletar","Bukit Merah","Kallang Basin","Braddell","MacPherson",
    "Outram","Telok Blangah","Downtown Core","Harbourfront","Balestier",
]

GAZETTEER_SORTED = sorted(set(GAZETTEER), key=len, reverse=True)

POLICY_LEXICON = [
    "URA Master Plan","GFA","Gross Floor Area","BTO","Build-to-Order",
    "HDB","EC","Executive Condominium","Government Land Sales","GLS",
    "Confirmed List","Reserve List","ABSD","cooling measures","TDSR",
    "Prime Location Public Housing","PLPH","SERS","CPF Housing Grant",
]

def first_location(text: str) -> Optional[str]:
    if not text: return None
    t = text.lower()
    for loc in GAZETTEER_SORTED:
        if re.search(r"\b" + re.escape(loc.lower()) + r"\b", t):
            return loc
    m = re.search(r"district\s+(\d+)", t)
    if m: return f"District {m.group(1)}"
    return None

def policy_flag(text: str) -> bool:
    if not isinstance(text, str): return False
    t = text.lower()
    return any(term.lower() in t for term in POLICY_LEXICON)

# ============== CELL 7: RAW → NORMALIZED (JSON/CSV) ========
def read_json_any(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".jsonl":
        rows=[]
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line=line.strip()
                if not line: continue
                try:
                    obj=json.loads(line)
                    rows.append(obj)
                except: pass
        df=pd.json_normalize(rows, max_level=2) if rows else pd.DataFrame()
        df["source_file"]=path.name; return df
    try:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            obj=json.load(f)
    except Exception:
        try:
            df=pd.read_json(str(path)); df["source_file"]=path.name; return df
        except: return pd.DataFrame({"_read_error":[str(path)], "source_file":[path.name]})
    if isinstance(obj, list):
        df=pd.json_normalize(obj, max_level=2); df["source_file"]=path.name; return df
    if isinstance(obj, dict):
        df=pd.json_normalize(obj, max_level=2); df["source_file"]=path.name; return df
    return pd.DataFrame(columns=["source_file"])

def infer_agency_from_path(p: Path) -> str:
    parts=[s.lower() for s in p.parts]
    for cand in ["hdb","mnd","mas","bca","sfa","sla"]:
        if cand in parts or p.name.lower().startswith(cand+"_"): return cand.upper()
    return "UNK"

def normalize_raw_to_combined(raw_root: str, out_csv: str) -> pd.DataFrame:
    paths = glob.glob(f"{raw_root}/**/*.json*", recursive=True)
    if not paths: raise FileNotFoundError(f"No JSON under {raw_root}")
    frames=[]
    for s in sorted(paths):
        p=Path(s); df=read_json_any(p)
        if df.empty: continue
        agency=infer_agency_from_path(p)
        if "agency" not in df.columns: df.insert(0,"agency",agency)
        else: df["agency"]=df["agency"].fillna(agency).replace("",agency)
        for c in ["text","title","timestamp","url","language","id","source"]:
            if c not in df.columns: df[c]=df.get(c, None)
        frames.append(df)
    if not frames: raise RuntimeError("No usable rows from raw JSON.")
    df=pd.concat(frames, ignore_index=True, sort=False)
    # harmonize title/text
    df["title"] = df.get("metadata.title", df.get("title","")).astype(str).apply(deep_clean_text)
    df["text"]  = df.get("text","").astype(str).apply(deep_clean_text)
    # drop empties
    df=df[(df["title"].str.len()>0) | (df["text"].str.len()>0)].copy()
    # de-dup
    def row_key(r):
        if pd.notna(r.get("url")) and str(r["url"]).strip():
            return "url:" + str(r["url"]).strip().lower()
        key=(str(r.get("title",""))+"||"+str(r.get("text",""))).lower()
        return "tt:" + hashlib.md5(key.encode("utf-8")).hexdigest()
    df["_k"]=df.apply(row_key, axis=1)
    df=df.sort_values(by=["timestamp","source_file"], na_position="last").drop_duplicates("_k")
    df.drop(columns=["_k"], inplace=True, errors="ignore")
    df["timestamp"]=df.get("timestamp").astype(str)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    logger.info(f"[NORMALIZE] {len(df)} rows → {out_csv}")
    return df

def build_combined_from_processed(processed_base: str, out_csv: str) -> pd.DataFrame:
    files=list(Path(processed_base).rglob("*.csv"))
    if not files: raise FileNotFoundError(f"No CSV under {processed_base}")
    frames=[]
    for f in files:
        try:
            d=pd.read_csv(f)
            if "agency" not in d.columns: d.insert(0,"agency", f.parent.name.upper())
            if "title" not in d.columns and "metadata.title" in d.columns: d["title"]=d["metadata.title"]
            if "text" not in d.columns and "cleaned_text" in d.columns: d["text"]=d["cleaned_text"]
            for c in ["id","timestamp","url","language","title","text"]:
                if c not in d.columns: d[c]=d.get(c, None)
            frames.append(d)
        except Exception as e:
            print("[WARN] skip", f, e)
    df=pd.concat(frames, ignore_index=True, sort=False)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    logger.info(f"[FALLBACK] {len(df)} rows → {out_csv}")
    return df

# ============== CELL 8: Regex flags & EntityRuler =========
def load_jsonl(path: Path):
    items=[]
    if not path.exists(): return items
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if ln:
                try: items.append(json.loads(ln))
                except: pass
    return items

def compile_regex_flags(regex_jsonl_path: str):
    pairs=[]
    p=Path(regex_jsonl_path)
    if not p.exists(): return pairs
    for it in load_jsonl(p):
        pat=it.get("pattern"); name=it.get("name","pattern")
        if not pat: continue
        try: pairs.append((f"rx_{name}", re.compile(pat, flags=re.I)))
        except re.error: pass
    return pairs

def apply_regex_flags(df: pd.DataFrame, pairs: List[tuple]) -> pd.DataFrame:
    if not pairs: return df
    tcol = next((c for c in ["clean_text","text","body"] if c in df.columns), None)
    if not tcol: return df
    ser=df[tcol].astype(str)
    for colname,patt in pairs:
        try: df[colname]=ser.str.contains(patt, na=False)
        except: df[colname]=False
    return df

def entityruler_entities(df: pd.DataFrame, patterns_path: str) -> pd.DataFrame:
    try:
        import spacy
        p=Path(patterns_path)
        if not p.exists():
            if "entities" not in df.columns: df["entities"]=[[] for _ in range(len(df))]
            return df
        nlp=spacy.blank("en")
        ruler=nlp.add_pipe("entity_ruler")
        ruler.from_disk(str(p))
        tcol = next((c for c in ["clean_text","text","body"] if c in df.columns), None)
        if not tcol:
            df["entities"]=[[] for _ in range(len(df))]; return df
        ents=[]
        for doc in nlp.pipe(df[tcol].astype(str).tolist(), batch_size=64):
            ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
        df["entities"]=ents
        df["entity"] = df["entities"].apply(lambda L: "; ".join(dict.fromkeys([e["text"] for e in L])) if isinstance(L,list) else None)
        df["entity_labels"] = df["entities"].apply(lambda L: "; ".join(dict.fromkeys([e["label"] for e in L])) if isinstance(L,list) else None)
        return df
    except Exception as e:
        print("[WARN] EntityRuler:", e)
        if "entities" not in df.columns: df["entities"]=[[] for _ in range(len(df))]
        return df

# ============== CELL 9: OpenAI Labeler =====================
@dataclass
class GPTConfig:
    model: str = MODEL
    temperature: float = 0.2
    max_tokens: int = 1200

class GPTLabeler:
    def __init__(self, config: GPTConfig):
        self.client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY",""))
        self.config = config

    @retry(reraise=True, stop=stop_after_attempt(3), wait=wait_exponential(min=1, max=10),
           retry=retry_if_exception_type(Exception))
    def _json_call(self, prompt: str) -> Dict:
        resp = self.client.chat.completions.create(
            model=self.config.model,
            messages=[{"role":"system","content":"You are a Singapore real estate expert. Return VALID JSON only."},
                      {"role":"user","content": prompt}],
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens,
        )
        txt = (resp.choices[0].message.content or "").strip()
        txt = re.sub(r"^```json\s*", "", txt, flags=re.I)
        txt = re.sub(r"\s*```$", "", txt)
        return json.loads(txt)

    def label_one(self, body: str) -> Dict:
        if not body or len(body.strip()) < 8:
            return self.empty()
        prompt = f"""Analyze this Singapore property-market text and return ONLY JSON:

{{
  "property_sentiment": {{"value":"positive|neutral|negative","confidence":0..1}},
  "aspect_based_sentiment": {{
    "price": {{"value":"positive|neutral|negative|unknown","confidence":0..1}},
    "demand": {{"value":"positive|neutral|negative|unknown","confidence":0..1}},
    "supply": {{"value":"positive|neutral|negative|unknown","confidence":0..1}},
    "policy": {{"value":"positive|neutral|negative|unknown","confidence":0..1}},
    "infrastructure": {{"value":"positive|neutral|negative|unknown","confidence":0..1}},
    "economic": {{"value":"positive|neutral|negative|unknown","confidence":0..1}}
  }},
  "named_entities": [{{"text":"...", "label":"LOCATION|AGENCY|METRIC|POLICY|ORG|PERSON"}}],
  "policy_impact": {{"mentioned": true|false, "terms": []}}
}}

Text:
{body[:3000]}
"""
        try:
            out = self._json_call(prompt)
        except Exception as e:
            logger.error(f"label_one error: {e}")
            out = self.empty()
        # sanitize
        if not isinstance(out, dict): out = self.empty()
        for top in ["property_sentiment","aspect_based_sentiment","policy_impact"]:
            if top not in out or not isinstance(out[top], (dict,list)):
                out[top] = {} if top!="named_entities" else []
        if "named_entities" not in out or not isinstance(out["named_entities"], list):
            out["named_entities"] = []
        return out

    def empty(self) -> Dict:
        return {
            "property_sentiment": {"value":"neutral","confidence":0.0},
            "aspect_based_sentiment": {},
            "named_entities": [],
            "policy_impact": {"mentioned": False, "terms": []}
        }

# ============== CELL 10: Robust body builder (FIXED) ======
def build_body_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Robustly build 'title', 'clean_text', and 'body' columns.
    Never returns scalars; always Series aligned to df.index.
    """
    df = df.copy()
    # Title
    if "metadata.title" in df.columns: title = df["metadata.title"]
    elif "title" in df.columns:       title = df["title"]
    else:                             title = pd.Series([""]*len(df), index=df.index)
    # Text
    if "cleaned_text" in df.columns:  text = df["cleaned_text"]
    elif "clean_text" in df.columns:  text = df["clean_text"]
    elif "text" in df.columns:        text = df["text"]
    elif "content" in df.columns:     text = df["content"]
    else:                             text = pd.Series([""]*len(df), index=df.index)
    # Clean
    title = title.astype(str).fillna("").apply(deep_clean_text)
    text  = text.astype(str).fillna("").apply(deep_clean_text)
    # Assign & body
    df["title"] = title
    df["clean_text"] = text
    df["body"] = (df["title"].fillna("") + "\n\n" + df["clean_text"].fillna("")).str.strip()
    return df

# ============== CELL 11: Flatteners & Enrichment ===========
def flatten_property(df: pd.DataFrame) -> pd.DataFrame:
    if "property_sentiment" not in df.columns:
        df["property_sentiment"] = "{}"
    def parse(x):
        d = safe_json_load(x)
        if isinstance(d, dict):
            return d.get("value"), d.get("confidence")
        return None, None
    vals = df["property_sentiment"].apply(parse)
    df["overall_sentiment"] = vals.apply(lambda x: x[0])
    df["overall_score"]     = vals.apply(lambda x: x[1])
    return df

def flatten_aspects(df: pd.DataFrame) -> pd.DataFrame:
    if "aspect_based_sentiment" not in df.columns:
        df["aspect_based_sentiment"] = "{}"
    aspects = ["price","demand","supply","policy","infrastructure","economic"]
    def parse(x):
        d = safe_json_load(x) or {}
        out = {}
        for a in aspects:
            v = d.get(a, {})
            out[a] = (pick_val(v), pick_score(v))
        return out
    parsed = df["aspect_based_sentiment"].apply(parse)
    for a in aspects:
        df[f"{a}_sentiment"] = parsed.apply(lambda m: m[a][0])
        df[f"{a}_score"]     = parsed.apply(lambda m: m[a][1])
    return df

def enrich_with_policy_lexicon(df: pd.DataFrame) -> pd.DataFrame:
    def count_terms(text):
        if not isinstance(text, str): return []
        t = text.lower()
        return [term for term in POLICY_LEXICON if term.lower() in t]
    hay = (df.get("title","").astype(str) + " " + df.get("clean_text","").astype(str))
    terms = hay.apply(count_terms)
    df["policy_terms"] = terms.apply(lambda L: "; ".join(L) if L else "")
    df["policy_term_count"] = terms.apply(lambda L: len(L))
    return df

def derive_fields(df: pd.DataFrame) -> pd.DataFrame:
    hay = (df.get("title","").astype(str) + " " + df.get("clean_text","").astype(str)).str.lower()
    df["policy_mentioned"] = hay.apply(policy_flag)

    def pick_location(row):
        # if you have any prefilled location column, honor it here first
        text = f"{row.get('title','')} {row.get('clean_text','')}"
        loc = first_location(text)
        return loc or "Singapore (national)"

    df["location"] = df.apply(pick_location, axis=1)

    ordered = [
        ("policy_sentiment","Policy"),
        ("price_sentiment","Price/Affordability"),
        ("demand_sentiment","Demand"),
        ("supply_sentiment","Supply"),
    ]
    def choose_aspect(row):
        for col,label in ordered:
            v = str(row.get(col,"") or "").lower()
            if v and v not in ("neutral","unknown","not_applicable","nan",""):
                return label
        return "General"
    df["aspect"] = df.apply(choose_aspect, axis=1)
    return df

def backfill_and_flags(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["thread_url","forum_name","entity","entity_labels"]:
        if c not in df.columns: df[c] = None
    if "timestamp" in df.columns:
        df["date"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["flag_Sentiment"] = df["overall_sentiment"].notna()
    df["flag_Location"]  = df["location"].notna()
    df["flag_Policy"]    = df.get("policy_mentioned", False)
    return df

def property_domain_enrich(df: pd.DataFrame) -> pd.DataFrame:
    """Run regex flags + EntityRuler + policy lexicon + derives + flags."""
    df = df.copy()
    # regex flags
    pairs = compile_regex_flags(REGEX_JSONL)
    df = apply_regex_flags(df, pairs)
    # entity ruler
    df = entityruler_entities(df, ENTITYRULER)
    # lexicon + derives
    df = enrich_with_policy_lexicon(df)
    df = derive_fields(df)
    df = backfill_and_flags(df)
    return df

# ============== CELL 12: Label only rows that need GPT =====
def label_subset_with_openai(df: pd.DataFrame) -> pd.DataFrame:
    """
    Labels only rows that need GPT:
      - property_sentiment empty OR
      - overall_confidence < 0.75 (if present from prior runs)
    Merges back and returns a full labeled DataFrame.
    """
    # NOTE: build_body_columns and column initialization are now handled before this function call

    # Optional agency filter
    if AGENCY_ONLY:
        df = df[df.get("agency","").astype(str).str.upper() == AGENCY_ONLY.upper()].copy()
        logger.info(f"[Filter] agency={AGENCY_ONLY} → rows={len(df)}")

    # Determine which rows need GPT
    def empty_or_low(ps):
        d = safe_json_load(ps)
        if not isinstance(d, dict): return True
        conf = d.get("confidence") or d.get("confidence_score") or 0.0
        try: conf = float(conf)
        except: conf = 0.0
        return conf < 0.75

    needs_gpt_mask = df["property_sentiment"].apply(lambda s: True if (s in ("", None) or (isinstance(s, float) and pd.isna(s))) else empty_or_low(s))
    df_for_gpt = df[needs_gpt_mask].copy()

    # Trim body to reduce tokens
    df_for_gpt["body"] = df_for_gpt["body"].str.slice(0, 3000)

    print(f"[Label] Sending to GPT: {len(df_for_gpt)} / {len(df)} rows")

    # Early exit: nothing to do
    if len(df_for_gpt) == 0:
        return df

    # Label the subset
    labeler = GPTLabeler(GPTConfig(model=MODEL))
    labeled_rows = []
    n = len(df_for_gpt)
    for i in range(0, n, BATCH_SIZE):
        batch = df_for_gpt.iloc[i:i+BATCH_SIZE]
        for idx, row in batch.iterrows():
            out = labeler.label_one(row["body"])
            labeled_rows.append({
                "idx": idx,
                "property_sentiment": json.dumps(out.get("property_sentiment", {}), ensure_ascii=False),
                "aspect_based_sentiment": json.dumps(out.get("aspect_based_sentiment", {}), ensure_ascii=False),
                "named_entities": json.dumps(out.get("named_entities", []), ensure_ascii=False),
                "policy_impact": json.dumps(out.get("policy_impact", {}), ensure_ascii=False),
                "labeling_timestamp": now_iso()
            })
        time.sleep(0.3)

    # Merge back
    upd = pd.DataFrame(labeled_rows).set_index("idx")
    for col in ["property_sentiment","aspect_based_sentiment","named_entities","policy_impact","labeling_timestamp"]:
        df.loc[upd.index, col] = upd[col]

    return df

# ============== CELL 13: RUN — Load → Clean → Label → Enrich ==============
print("="*80, "\n🚀 STARTING PROPERTY DOMAIN PIPELINE\n", "="*80)

# Discover RAW or fallback
raw_candidate = RAW_BASE
if not Path(raw_candidate).exists():
    for cand in [f"{BASE}/raw", f"{BASE}/data/raw", f"{BASE}/government_websites"]:
        if Path(cand).exists():
            raw_candidate = cand; break

try:
    df_combined = normalize_raw_to_combined(raw_candidate, RAW_COMBINED)
    print(f"✅ Combined from RAW: {len(df_combined)}")
except Exception as e:
    print(f"⚠️ RAW load failed: {e} — falling back to processed")
    df_combined = build_combined_from_processed(PROCESSED_BASE, RAW_COMBINED)
    print(f"✅ Combined from processed: {len(df_combined)}")

# Clean deep
print("\n[1/3] Cleaning...")
df_clean = build_body_columns(df_combined)  # uses deep_clean_text internally

# === COLUMN GUARD (add right before label_subset_with_openai) ===
# Make sure body columns exist first
df_clean = build_body_columns(df_clean)

# Ensure JSON columns exist (first run may not have them)
# Use "{}" for object-JSON columns and "[]" for array-JSON columns
DEFAULTS = {
    "property_sentiment": "{}",
    "aspect_based_sentiment": "{}",
    "policy_impact": "{}",
    "named_entities": "[]",
    "labeling_timestamp": ""
}
for col, default in DEFAULTS.items():
    if col not in df_clean.columns:
        df_clean[col] = default
    else:
        # replace NaN with safe default
        df_clean[col] = df_clean[col].fillna(default)

# If you want to run only a sample on first run:
if MAX_RECORDS and MAX_RECORDS > 0:
    df_clean = df_clean.head(MAX_RECORDS).copy()
    print(f"[Limit] Using first {len(df_clean)} rows for this test run.")


# Label only rows that need it
print("\n[2/3] Labeling (subset only)...")
df_labeled = label_subset_with_openai(df_clean)

# Enrich & save
print("\n[3/3] Enriching & Saving...")
df_flat = df_labeled.copy()
df_flat = flatten_property(df_flat)
df_flat = flatten_aspects(df_flat)
df_flat = property_domain_enrich(df_flat)

# Save per-agency + ALL
def save_split_by_agency(df: pd.DataFrame, outdir: str, suffix: str):
    out = Path(outdir); out.mkdir(parents=True, exist_ok=True)
    paths=[]
    if "agency" in df.columns:
        for ag in sorted(df["agency"].dropna().unique()):
            sub=df[df["agency"]==ag].copy()
            p=out/f"{ag}_{suffix}.csv"; sub.to_csv(p, index=False); paths.append(p)
    p_all=out/f"ALL_{suffix}.csv"; df.to_csv(p_all, index=False); paths.append(p_all)
    return paths

labeled_paths  = save_split_by_agency(df_labeled, OUTPUT_DIR, suffix="labeled")
enriched_paths = save_split_by_agency(df_flat,   OUTPUT_DIR, suffix="labeled_enriched")

print("\n✅ Saved:")
for p in labeled_paths+enriched_paths: print(" ", p)

print("\n🎉 Done. Review the enriched file(s) above.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config: 
  MODEL=gpt-4o 
  MAX_RECORDS=ALL 
  AGENCY_ONLY=ALL
🚀 STARTING PROPERTY DOMAIN PIPELINE
✅ Combined from RAW: 59

[1/3] Cleaning...

[2/3] Labeling (subset only)...
[Label] Sending to GPT: 59 / 59 rows

[3/3] Enriching & Saving...

✅ Saved:
  /content/drive/MyDrive/PropInsight/labeled/BCA_labeled.csv
  /content/drive/MyDrive/PropInsight/labeled/MAS_labeled.csv
  /content/drive/MyDrive/PropInsight/labeled/MND_labeled.csv
  /content/drive/MyDrive/PropInsight/labeled/SFA_labeled.csv
  /content/drive/MyDrive/PropInsight/labeled/SLA_labeled.csv
  /content/drive/MyDrive/PropInsight/labeled/ALL_labeled.csv
  /content/drive/MyDrive/PropInsight/labeled/BCA_labeled_enriched.csv
  /content/drive/MyDrive/PropInsight/labeled/MAS_labeled_enriched.csv
  /content/drive/MyDrive/PropInsight/labeled/MND_labeled_enriched.csv
  /content/drive/MyDrive/PropInsight/labeled/